# 03 — Exploratory Data Analysis (EDA)

> **Objective:** Analyze temperature trends, precipitation distribution, variable correlations, and monthly seasonality. Export all charts to `reports/`.

---

## 1. Setup

Import visualization libraries and configure output paths.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import (
    load_raw_weather,
    add_region_column,
    get_temperature_column,
    get_precipitation_column,
)

reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Data

Load the raw CSV for EDA (keeps physical units and categorical columns intact).

In [ ]:
# Load raw CSV for EDA (keeps physical units and categorical columns intact)
df = load_raw_weather(project_root)
df = add_region_column(df)

temp_col = get_temperature_column(df)
precip_col = get_precipitation_column(df)

# Prepare EDA frame with time features
eda = df.dropna(subset=["last_updated", temp_col]).copy()
eda["year_month"] = eda["last_updated"].dt.to_period("M").astype(str)
eda["month"] = eda["last_updated"].dt.month

print("EDA shape:", eda.shape)
print("Temperature column:", temp_col)
print("Precipitation column:", precip_col)
eda[["last_updated", "region", temp_col, precip_col]].head()

## 3. Temperature Trend by Region

Monthly average temperature over time, grouped by geographic region.

In [ ]:
temp_monthly = (
    eda.groupby(["year_month", "region"], as_index=False)[temp_col]
    .mean()
    .sort_values("year_month")
)

fig1 = px.line(
    temp_monthly,
    x="year_month",
    y=temp_col,
    color="region",
    title="Average Temperature Trend by Region (Monthly)",
    labels={"year_month": "Year-Month", temp_col: "Avg Temperature", "region": "Region"},
)
fig1.update_layout(legend_title_text="Region")
_html_out = reports_dir / "eda_temperature_trend_by_region.html"
fig1.write_html(_html_out)
print("Saved:", _html_out)
# Open the HTML file in a browser if inline Plotly is unavailable (requires nbformat in Jupyter).
try:
    fig1.show()
except ValueError as exc:
    if "nbformat" in str(exc).lower():
        print("Plotly inline skipped. Install: pip install 'nbformat>=4.2.0' — or open the HTML file above.")
    else:
        raise

**Interpretation:**
- Compare regional curves to identify warmer/cooler regimes and seasonal shifts.
- Large divergence may indicate different climate baselines or sampling imbalance.

## 4. Precipitation Distribution by Region

Box plots showing precipitation spread across geographic regions.

In [ ]:
precip_sample = eda[["region", precip_col]].dropna().copy()
# Cap extreme tail for readability in boxplot visualization.
upper_cap = precip_sample[precip_col].quantile(0.99)
precip_sample[precip_col] = precip_sample[precip_col].clip(upper=upper_cap)

plt.figure(figsize=(12, 6))
order = precip_sample.groupby("region")[precip_col].median().sort_values(ascending=False).index
sns.boxplot(data=precip_sample, x="region", y=precip_col, order=order)
plt.title("Precipitation Distribution by Region (Capped at 99th percentile)")
plt.xlabel("Region")
plt.ylabel("Precipitation")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(reports_dir / "eda_precipitation_distribution_by_region.png", dpi=150)
plt.show()

**Interpretation:**
- Median and spread differences indicate stable vs volatile rainfall patterns.
- Wide IQR and long tails suggest frequent extreme precipitation events.

## 5. Correlation Analysis

Static heatmap (Seaborn) and interactive 3D surface (Plotly) of variable correlations.

In [ ]:
numeric_cols = eda.select_dtypes(include=[np.number]).columns.tolist()

# Keep a focused set of interpretable weather metrics when available.
priority_vars = [
    "temperature_celsius",
    "humidity",
    "wind_kph",
    "pressure_mb",
    "precip_mm",
    "cloud",
    "uv",
    "gust_kph",
]
selected_numeric = [c for c in priority_vars if c in numeric_cols]
if len(selected_numeric) < 4:
    selected_numeric = numeric_cols[:12]

corr = eda[selected_numeric].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap of Climate Variables")
plt.tight_layout()
plt.savefig(reports_dir / "eda_correlation_heatmap.png", dpi=150)
plt.show()
plt.close()

# 3D surface: same correlation matrix (interactive rotation in exported HTML)
labels = [str(c)[:24] for c in corr.columns]
z = corr.values.astype(float)
n = z.shape[0]
fig3d = go.Figure(
    data=[
        go.Surface(
            z=z,
            colorscale="RdBu",
            cmin=-1,
            cmax=1,
            colorbar=dict(title="r"),
        )
    ]
)
fig3d.update_layout(
    title="Correlation matrix — 3D surface (z = Pearson r)",
    scene=dict(
        xaxis=dict(
            tickmode="array",
            tickvals=list(range(n)),
            ticktext=labels,
            title="Column index",
        ),
        yaxis=dict(
            tickmode="array",
            tickvals=list(range(n)),
            ticktext=labels,
            title="Row index",
        ),
        zaxis=dict(title="Correlation"),
        aspectmode="manual",
        aspectratio=dict(x=1, y=1, z=0.55),
    ),
    margin=dict(l=0, r=0, b=0, t=50),
    height=700,
)
out3d = reports_dir / "eda_correlation_surface_3d.html"
fig3d.write_html(out3d)
print("Saved:", out3d)
try:
    fig3d.show()
except ValueError as exc:
    if "nbformat" in str(exc).lower():
        print("Open the HTML file to rotate the 3D surface interactively.")
    else:
        raise

**Interpretation:**
- Positive/negative correlation blocks highlight coupled weather dynamics.
- Strong correlations guide feature selection and multicollinearity checks.
- The 3D surface shows correlation "ridges" (open HTML for interactive rotation).

## 6. Monthly Seasonality

Average temperature profile across calendar months.

In [ ]:
seasonality = eda.groupby("month", as_index=False)[temp_col].mean().sort_values("month")

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.figure(figsize=(10, 5))
sns.lineplot(data=seasonality, x="month", y=temp_col, marker="o")
plt.title("Average Monthly Temperature Seasonality")
plt.xlabel("Month")
plt.ylabel("Average Temperature")
plt.xticks(ticks=range(1, 13), labels=month_labels)
plt.tight_layout()
plt.savefig(reports_dir / "eda_monthly_temperature_seasonality.png", dpi=150)
plt.show()

**Interpretation:**
- Peaks and troughs indicate expected seasonal cycles.
- Use this profile to justify yearly seasonality terms in forecasting models.

In [ ]:
# Export summary
exported_files = [
    reports_dir / "eda_temperature_trend_by_region.html",
    reports_dir / "eda_precipitation_distribution_by_region.png",
    reports_dir / "eda_correlation_heatmap.png",
    reports_dir / "eda_correlation_surface_3d.html",
    reports_dir / "eda_monthly_temperature_seasonality.png",
]

for f in exported_files:
    print(f"{f.name}: {'OK' if f.exists() else 'MISSING'}")